In [1]:
from supabase import create_client, Client
from dotenv import load_dotenv
import os
import pandas as pd

# Load the .env file from the parent directory
load_dotenv("../.env")

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")


supabase_client: Client = create_client(url, key)

if supabase_client:
    print("Supabase client created successfully")
else:
    print("Failed to create Supabase client")




Supabase client created successfully


In [2]:
import time
import os
import numpy as np
import pandas as pd
import glob

def extract_date_from_filename(filepath):
    """Extracts date from a filename format like 'YYYYMMDD_name.ext' or 'YYYYMM_name.ext' and returns 'YYYY-MM-DD'."""
    filename = os.path.basename(filepath)
    date_part = filename.split("_")[0]
    
    # Check if we have at least 6 characters (YYYYMM)
    if len(date_part) >= 6:
        year = date_part[:4]
        month = date_part[4:6]
        
        # Try to get the day (characters 6 and 7), otherwise default to "01"
        day = date_part[6:8] if len(date_part) >= 8 else "01"
        
        # Just in case day is empty after slicing
        if not day:
            day = "01"
            
        return f"{year}-{month}-{day}"
        
    return None

def process_file_data(filepath, expected_columns):
    """Loads a CSV/GZ file, formats it, and filters to expected columns."""
    # 1. Load data
    df = pd.read_csv(filepath, compression="gzip", sep="|", encoding="latin-1")
    
    # 2. Standardize column names to lowercase
    df.columns = df.columns.str.lower()
    
    # 3. Extract and add the data_date
    date_string = extract_date_from_filename(filepath)
    if date_string:
        # We can just assign the string directly, which avoids unnecessary datetime conversion
        df["data_date"] = date_string 
        
    # 4. Filter only columns that exist in both the dataframe and the expected schema
    cols_to_keep = [col for col in df.columns if col in expected_columns]
    df = df[cols_to_keep]
    
    # 5. Replace NaN/NaT values with None for JSON/Supabase compatibility
    df_clean = df.replace({np.nan: None})
    
    return df_clean

# def upload_to_supabase(df, table_name, supabase_client, chunk_size=1000):
#     """Uploads a dataframe to a Supabase table in chunks using upsert."""
#     records = df.to_dict(orient="records")
#     total_chunks = (len(records) + chunk_size - 1) // chunk_size

#     print(f"Uploading {len(records)} records to '{table_name}' in {total_chunks} chunks")
    
#     for i in range(0, len(records), chunk_size):
#         chunk = records[i : i + chunk_size]
        
#         # Push the chunk to the table using upsert instead of insert
#         response = supabase_client.table(table_name).upsert(chunk).execute()
        
    
#     print(f"Completed {i//chunk_size + 1} of {total_chunks} chunks")
        
#     print(f"Upload to '{table_name}' complete!\n\n")

from concurrent.futures import ThreadPoolExecutor, as_completed

def upload_to_supabase(df, table_name, supabase_client, chunk_size=1000, max_workers=6, max_retry_rounds=5):
    records = df.to_dict(orient="records")
    all_chunks = {i: records[j : j + chunk_size] for i, j in enumerate(range(0, len(records), chunk_size))}
    total = len(all_chunks)
    print(f"Uploading {len(records)} records to '{table_name}' in {total} chunks ({max_workers} threads)")

    def _upsert(idx, chunk):
        for attempt in range(3):
            try:
                supabase_client.table(table_name).upsert(chunk).execute()
                return idx, True
            except Exception as e:
                if attempt == 2:
                    return idx, False
                time.sleep(2 ** attempt)

    pending = dict(all_chunks)
    for round_num in range(max_retry_rounds):
        if round_num > 0:
            wait = min(30, 5 * round_num)
            print(f"  Retry round {round_num}/{max_retry_rounds - 1}: {len(pending)} chunks remaining (waiting {wait}s)...")
            time.sleep(wait)

        failed = {}
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(_upsert, i, c): i for i, c in pending.items()}
            done_count = 0
            for future in as_completed(futures):
                idx, ok = future.result()
                done_count += 1
                if not ok:
                    failed[idx] = pending[idx]
                if done_count % 50 == 0 or done_count == len(pending):
                    print(f"  Progress: {done_count}/{len(pending)}")

        if not failed:
            break
        pending = failed

    if failed:
        raise RuntimeError(
            f"FAILED: {len(failed)}/{total} chunks could not be uploaded to '{table_name}' "
            f"after {max_retry_rounds} rounds: {sorted(failed.keys())}"
        )
    print(f"Upload to '{table_name}' complete!")

# --- Define Expected Schemas ---
FORMULARY_COLUMNS = [
    "formulary_id", "formulary_version", "contract_year", "rxcui", "ndc", 
    "tier_level_value", "quantity_limit_yn", "quantity_limit_amount", 
    "quantity_limit_days", "prior_authorization_yn", "step_therapy_yn", 
    "selected_drug_yn", "data_date"
]

PLANS_COLUMNS = [
    "contract_id", "plan_id", "segment_id", "contract_name", "plan_name", 
    "formulary_id", "premium", "deductible", "ma_region_code", "pdp_region_code", 
    "state", "county_code", "snp", "plan_suppressed_yn", "data_date"
]

COSTS_COLUMNS = [
    "contract_id", "plan_id", "segment_id", "coverage_level", "tier",
    "days_supply", "cost_type_pref", "cost_amt_pref", "cost_min_amt_pref",
    "cost_max_amt_pref", "cost_type_nonpref", "cost_amt_nonpref",
    "cost_min_amt_nonpref", "cost_max_amt_nonpref", "cost_type_mail_pref",
    "cost_amt_mail_pref", "cost_min_amt_mail_pref", "cost_max_amt_mail_pref",
    "cost_type_mail_nonpref", "cost_amt_mail_nonpref", "cost_min_amt_mail_nonpref",
    "cost_max_amt_mail_nonpref", "tier_specialty_yn", "ded_applies_yn",
    "data_date"
]

# ==========================================
# Example 1: Uploading Formulary
# ==========================================
for formulary_file in glob.glob("formulary/*.txt.gz"):
    print(f"Processing {formulary_file}...")
    df_formulary = process_file_data(formulary_file, FORMULARY_COLUMNS)
    upload_to_supabase(df_formulary, "formulary", supabase_client)
    
    # Sleep for 10 second to avoid overwhelming the server
    time.sleep(10)

# ==========================================
# Example 2: Uploading Plans
# ==========================================
for plans_file in glob.glob("plans/*.txt.gz"):
    print(f"Processing {plans_file}...")
    df_plans = process_file_data(plans_file, PLANS_COLUMNS)
    upload_to_supabase(df_plans, "plans", supabase_client)

    # Sleep for 10 second to avoid overwhelming the server
    time.sleep(10)

# ==========================================
# Example 3: Uploading Costs
# ==========================================
for costs_file in glob.glob("costs/*.txt.gz"):
    print(f"Processing {costs_file}...")
    df_costs = process_file_data(costs_file, COSTS_COLUMNS)
    upload_to_supabase(df_costs, "costs", supabase_client)

    # Sleep for 10 second to avoid overwhelming the server
    time.sleep(10)


Processing formulary/20260122_formulary.txt.gz...
Uploading 1118219 records to 'formulary' in 1119 chunks (6 threads)
  Progress: 50/1119
  Progress: 100/1119
  Progress: 150/1119
  Progress: 200/1119
  Progress: 250/1119
  Progress: 300/1119
  Progress: 350/1119
  Progress: 400/1119
  Progress: 450/1119
  Progress: 500/1119
  Progress: 550/1119
  Progress: 600/1119
  Progress: 650/1119
  Progress: 700/1119
  Progress: 750/1119
  Progress: 800/1119
  Progress: 850/1119
  Progress: 900/1119
  Progress: 950/1119
  Progress: 1000/1119
  Progress: 1050/1119
  Progress: 1100/1119
  Progress: 1119/1119
  Retry round 1/4: 35 chunks remaining (waiting 5s)...
  Progress: 35/35
Upload to 'formulary' complete!
Processing formulary/20260219_formulary.txt.gz...
Uploading 1122733 records to 'formulary' in 1123 chunks (6 threads)
  Progress: 50/1123
  Progress: 100/1123
  Progress: 150/1123
  Progress: 200/1123
  Progress: 250/1123
  Progress: 300/1123
  Progress: 350/1123
  Progress: 400/1123
  Prog

/var/folders/fd/6q4zq38d5tq23ls0ffnqq4_w0000gn/T/ipykernel_76385/380618259.py:31: DtypeWarning: Columns (0: COUNTY_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, compression="gzip", sep="|", encoding="latin-1")


Uploading 112638 records to 'plans' in 113 chunks (6 threads)
  Progress: 50/113
  Progress: 100/113
  Progress: 113/113
  Retry round 1/4: 5 chunks remaining (waiting 5s)...
  Progress: 5/5
Upload to 'plans' complete!
Processing plans/20260122_plans.txt.gz...


/var/folders/fd/6q4zq38d5tq23ls0ffnqq4_w0000gn/T/ipykernel_76385/380618259.py:31: DtypeWarning: Columns (0: COUNTY_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, compression="gzip", sep="|", encoding="latin-1")


Uploading 112653 records to 'plans' in 113 chunks (6 threads)
  Progress: 50/113
  Progress: 100/113
  Progress: 113/113
Upload to 'plans' complete!
Processing costs/20260122_costs.txt.gz...
Uploading 173155 records to 'costs' in 174 chunks (6 threads)
  Progress: 50/174
  Progress: 100/174
  Progress: 150/174
  Progress: 174/174
Upload to 'costs' complete!
Processing costs/20260219_costs.txt.gz...
Uploading 173155 records to 'costs' in 174 chunks (6 threads)
  Progress: 50/174
  Progress: 100/174
  Progress: 150/174
  Progress: 174/174
Upload to 'costs' complete!


In [ ]:
# df_costs = pd.read_csv("costs/20260122_costs.txt.gz", compression="gzip", sep="|", encoding="latin-1")
# df_costs.head()


,CONTRACT_ID,PLAN_ID,SEGMENT_ID,COVERAGE_LEVEL,TIER,DAYS_SUPPLY,COST_TYPE_PREF,COST_AMT_PREF,COST_MIN_AMT_PREF,COST_MAX_AMT_PREF,...,COST_TYPE_MAIL_PREF,COST_AMT_MAIL_PREF,COST_MIN_AMT_MAIL_PREF,COST_MAX_AMT_MAIL_PREF,COST_TYPE_MAIL_NONPREF,COST_AMT_MAIL_NONPREF,COST_MIN_AMT_MAIL_NONPREF,COST_MAX_AMT_MAIL_NONPREF,TIER_SPECIALTY_YN,DED_APPLIES_YN
0,H0028,7,0,0,1,1,0,0.0,0,0.0,...,1,0.0,0,0.0,1,10.0,0.0,0.0,N,N
1,H0028,7,0,0,1,2,0,0.0,0,0.0,...,1,0.0,0,0.0,1,30.0,0.0,0.0,N,N
2,H0028,7,0,0,2,1,0,0.0,0,0.0,...,1,0.0,0,0.0,1,20.0,0.0,0.0,N,N
3,H0028,7,0,0,2,2,0,0.0,0,0.0,...,1,0.0,0,0.0,1,60.0,0.0,0.0,N,N
4,H0028,7,0,1,1,1,0,0.0,0,0.0,...,1,0.0,0,0.0,1,10.0,0.0,0.0,N,N


In [ ]:
# df_plans = pd.read_csv("plans/20260122_plans.txt.gz", compression="gzip", sep="|", encoding="latin-1")
# df_plans.head()

/var/folders/fd/6q4zq38d5tq23ls0ffnqq4_w0000gn/T/ipykernel_58388/2362362134.py:1: DtypeWarning: Columns (0: COUNTY_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df_plans = pd.read_csv("plans/20260122_plans.txt.gz", compression="gzip", sep="|", encoding="latin-1")


,CONTRACT_ID,PLAN_ID,SEGMENT_ID,CONTRACT_NAME,PLAN_NAME,FORMULARY_ID,PREMIUM,DEDUCTIBLE,MA_REGION_CODE,PDP_REGION_CODE,STATE,COUNTY_CODE,SNP,PLAN_SUPPRESSED_YN
0,H0028,7,0,"CHA HMO, INC.",Humana Gold Plus SNP-DE H0028-007 (HMO D-SNP),26408,35.6,615,,,NE,28100,2,N
1,H0028,7,0,"CHA HMO, INC.",Humana Gold Plus SNP-DE H0028-007 (HMO D-SNP),26408,35.6,615,,,NE,28110,2,N
2,H0028,7,0,"CHA HMO, INC.",Humana Gold Plus SNP-DE H0028-007 (HMO D-SNP),26408,35.6,615,,,NE,28120,2,N
3,H0028,7,0,"CHA HMO, INC.",Humana Gold Plus SNP-DE H0028-007 (HMO D-SNP),26408,35.6,615,,,NE,28180,2,N
4,H0028,7,0,"CHA HMO, INC.",Humana Gold Plus SNP-DE H0028-007 (HMO D-SNP),26408,35.6,615,,,NE,28190,2,N
